# Specialize Benchmarks
Compare **normal** (`@finch_kernel`) vs **specialized** (`specialize()`) Finch dense tensors.

In [23]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.develop(PackageSpec(; path=joinpath(@__DIR__, "..")))
Pkg.instantiate()

  Activating project at `~/Finch.jl/benchmark`
   Resolving package versions...
     Project No packages added to or removed from `~/Finch.jl/benchmark/Project.toml`
    Manifest No packages added to or removed from `~/Finch.jl/benchmark/Manifest.toml`


In [24]:
using Finch
using BenchmarkTools

## Kernel Definitions - Normal AOT

In [25]:
eval(let
    A = Tensor(Dense(Element(0.0)))
    s = Tensor(Element(0.0))
    @finch_kernel function vecsum(s, A)
        s .= 0
        for i in _
            s[] += A[i]
        end
        return s
    end
end)

eval(let
    A = Tensor(Dense(Dense(Element(0.0))))
    s = Tensor(Element(0.0))
    @finch_kernel function matsum(s, A)
        s .= 0
        for j in _, i in _
            s[] += A[i, j]
        end
        return s
    end
end)

eval(let
    A = Tensor(Dense(Dense(Element(0.0))))
    x = Tensor(Dense(Element(0.0)))
    y = Tensor(Dense(Element(0.0)))
    @finch_kernel function matvec(y, A, x)
        y .= 0
        for j in _, i in _
            y[i] += A[i, j] * x[j]
        end
        return y
    end
end)

eval(let
    A = Tensor(Dense(SparseList(Element(0.0))))
    x = Tensor(Dense(Element(0.0)))
    y = Tensor(Dense(Element(0.0)))
    @finch_kernel function spmv(y, A, x)
        y .= 0
        for j in _, i in _
            y[i] += A[i, j] * x[j]
        end
        return y
    end
end)

spmv (generic function with 1 method)

## Kernel Definitions - Specialized AOT

Specialized kernels via @finch_kernel (dimensions baked into type). Each specialize()'d size is a distinct type, so @finch_kernel generates a separate method per size. Julia dispatch selects the right one.

In [26]:
# vecsum_spec: N = 8, 100, 10_000
for N in [8, 100, 10_000]
    eval(let
        sA = specialize(Tensor(Dense(Element(0.0)), rand(N)))
        s  = Tensor(Element(0.0))
        @finch_kernel function vecsum_spec(s, sA)
            s .= 0
            for i in _
                s[] += sA[i]
            end
            return s
        end
    end)
end

# matsum_spec: (4,4), (10,10), (100,100)
for (M, N) in [(4, 4), (10, 10), (100, 100)]
    eval(let
        sA = specialize(Tensor(Dense(Dense(Element(0.0))), rand(M, N)))
        s  = Tensor(Element(0.0))
        @finch_kernel function matsum_spec(s, sA)
            s .= 0
            for j in _, i in _
                s[] += sA[i, j]
            end
            return s
        end
    end)
end

# matvec_spec: N = 8, 32, 256
for N in [8, 32, 256]
    eval(let
        sA = specialize(Tensor(Dense(Dense(Element(0.0))), rand(N, N)))
        sx = specialize(Tensor(Dense(Element(0.0)), rand(N)))
        y  = Tensor(Dense(Element(0.0)))
        @finch_kernel function matvec_spec(y, sA, sx)
            y .= 0
            for j in _, i in _
                y[i] += sA[i, j] * sx[j]
            end
            return y
        end
    end)
end

# spmv_spec: (100,10), (1000,10), (10_000,10)
for (N, nnz_per_col) in [(100, 10), (1000, 10), (10_000, 10)]
    density = nnz_per_col / N
    eval(let
        sA = specialize(Tensor(Dense(SparseList(Element(0.0))), fsprand(N, N, density)))
        x  = Tensor(Dense(Element(0.0)), rand(N))
        y  = Tensor(Dense(Element(0.0)))
        @finch_kernel function spmv_spec(y, sA, x)
            y .= 0
            for j in _, i in _
                y[i] += sA[i, j] * x[j]
            end
            return y
        end
    end)
end

---
## 1. Vector Sum
### N = 8

In [27]:
let N = 8
    A  = Tensor(Dense(Element(0.0)), rand(N))
    sA = specialize(A)
    s  = Tensor(Element(0.0))
    println("── normal ──")
    display(@benchmark vecsum($s, $A))
    println("\n── specialized ──")
    display(@benchmark vecsum_spec($s, $sA))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  4.726 ns … 8.310 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.754 ns             ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.869 ns ± 0.270 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▇▁    ▃▆▂      ▁      ▁▃       ▂                         ▁
  ███▇▆▆▆███▆▄▂▂▃▅█▅▃▂▂▃▃██▇▅▅▃▄▃██▆▄▄▃▄▄▆▇▄▄▄▄▄▅▆▅▅▆▆▇▆▄▅▅ █
  4.73 ns     Histogram: log(frequency) by time     6.11 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  4.727 ns … 16.043 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.745 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.799 ns ±  0.199 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▇█▆▃             ▁▅▆▁                                      ▂
  ████▇▇▇▅▆▇█▅▄▅▅▅▄████▇▆▆▅▃▂▂▂▄▃▃▂▂▃▄▅▅▄▆▅▄▅▅▄▂▄▂▄▄▆▄▅▅▆▆▇▅ █
  4.73 ns      Histogram: log(frequency) by time     5.33 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### N = 100

In [28]:
let N = 100
    A  = Tensor(Dense(Element(0.0)), rand(N))
    sA = specialize(A)
    s  = Tensor(Element(0.0))
    println("── normal ──")
    display(@benchmark vecsum($s, $A))
    println("\n── specialized ──")
    display(@benchmark vecsum_spec($s, $sA))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  6.058 ns … 23.232 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     6.102 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   6.347 ns ±  0.632 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▆▃▁▁▄▃▁▁    ▁ ▂ ▁                                       ▂ ▁
  █████████▇█████████▇▆▆▆▇▆▄▆▅▆▇▇▇▇▆▆▆▆▄▅▃▄▄▅▄▅▅▅▅▆▆▆▆▅▆▆▆██ █
  6.06 ns      Histogram: log(frequency) by time     8.77 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  5.339 ns … 18.119 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     5.424 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.544 ns ±  0.273 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▃█▇▂                                                      
  ▃▅████▇▄▄▄▄▃▂▂▂▂▃▃▃▂▂▂▂▃▄▅▄▃▂▂▂▂▂▂▂▃▃▄▄▄▃▂▂▂▂▂▂▁▁▂▂▂▂▂▂▂▃▂ ▃
  5.34 ns        Histogram: frequency by time        6.25 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### N = 10,000

In [29]:
let N = 1_000_000
    A  = Tensor(Dense(Element(0.0)), rand(N))
    sA = specialize(A)
    s  = Tensor(Element(0.0))
    println("── normal ──")
    display(@benchmark vecsum($s, $A))
    println("\n── specialized ──")
    display(@benchmark vecsum_spec($s, $sA))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  147.121 μs … 325.485 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     155.770 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   156.303 μs ±   3.578 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                       ▂▆█▇▇▄▃▁▁▁▁      ▂▂▃▂▁                    
  ▂▁▁▁▁▁▁▁▂▂▁▂▁▁▁▁▁▁▂▄▇█████████████▇▇▇██████▇▆▆▅▄▄▃▃▃▃▃▃▃▃▃▂▂▂ ▄
  147 μs           Histogram: frequency by time          164 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  150.186 μs … 321.327 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     152.485 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   152.845 μs ±   2.288 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

              ▂▅▆▆▇█▇▆▃                                          
  ▂▁▁▁▁▁▂▁▂▂▃▆██████████▄▃▃▃▃▄▄▄▄▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂ ▃
  150 μs           Histogram: frequency by time          158 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

---
## 2. Matrix Sum
### 4 × 4

In [30]:
let M = 4, N = 4
    A  = Tensor(Dense(Dense(Element(0.0))), rand(M, N))
    sA = specialize(A)
    s  = Tensor(Element(0.0))
    println("── normal ──")
    display(@benchmark matsum($s, $A))
    println("\n── specialized ──")
    display(@benchmark matsum_spec($s, $sA))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 999 evaluations per sample.
 Range (min … max):  8.152 ns … 10.837 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     8.190 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   8.253 ns ±  0.251 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ██▆▄▄▄▃▂▂   ▁                                              ▂
  █████████▇▅▆██▆▆▆▃▄▅▁▄▇▇▆▅▃▅▄▃▁▄▄▅▅▆▆▆▆▆▅▆▅▅▅▆▅▄▅▃▅▅▄▅▅▅▇█ █
  8.15 ns      Histogram: log(frequency) by time     9.76 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  5.484 ns … 23.815 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     5.509 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.556 ns ±  0.238 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▇█▇▅▂          ▁▄▄▁                                        ▁
  ██████▆▆▆▆▆▆▅▆▆████▇▆▆▅▅▅▄▄▅▄▆█▇▃▄▄▄▅▄▄▄▅▃▅▄▄▄▄▅▄▅▄▅▄▅▄▄▄▅ █
  5.48 ns      Histogram: log(frequency) by time     6.29 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### 100 × 100

In [31]:
let M = 100, N = 100
    A  = Tensor(Dense(Dense(Element(0.0))), rand(M, N))
    sA = specialize(A)
    s  = Tensor(Element(0.0))
    println("── normal ──")
    display(@benchmark matsum($s, $A))
    println("\n── specialized ──")
    display(@benchmark matsum_spec($s, $sA))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 148 evaluations per sample.
 Range (min … max):  681.331 ns … 908.399 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     693.845 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   698.215 ns ±  10.590 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

            ▂█▅▆▄▂▄▄▂▂▁▁▁▁▁            ▃▁                     ▁ ▂
  ▃▃▁▃▁▁▁▁▆██████████████████████████████▇█▆▅▅▆▅▅▅▅▅▄▄▅▅▅▅▃▅▄▄█ █
  681 ns        Histogram: log(frequency) by time        745 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.138 μs …  2.470 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.144 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.149 μs ± 21.825 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▄▇█▅▃                                                      
  ▄█████▅▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▄▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂ ▃
  1.14 μs        Histogram: frequency by time        1.23 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

---
## 3. Dense Matvec
### N = 8

In [32]:
let N = 8
    A  = Tensor(Dense(Dense(Element(0.0))), rand(N, N))
    sA = specialize(A)
    x  = Tensor(Dense(Element(0.0)), rand(N))
    sx = specialize(x)
    y  = Tensor(Dense(Element(0.0)))
    println("── normal ──")
    display(@benchmark matvec($y, $A, $x))
    println("\n── specialized ──")
    display(@benchmark matvec_spec($y, $sA, $sx))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 997 evaluations per sample.
 Range (min … max):  20.653 ns … 75.355 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     20.928 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   21.329 ns ±  0.960 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ██▅▁                                                        
  ▄████▃▆▄▃▄▃▂▂▁▁▃▄▃▃▂▁▂▁▁▂▁▁▁▁▁▃▄▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁ ▂
  20.7 ns         Histogram: frequency by time          24 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 998 evaluations per sample.
 Range (min … max):  15.689 ns … 29.165 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     15.720 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   15.862 ns ±  0.505 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▃          ▃▁                                              ▁
  ██▆▆▇▆▆▇▆▆▆▆██▄▅▇▇▇▆▆▅▅▅▄▆▅▃▄▄▄▄▃▅▄▄▄▄▄▃▄▅▄▄▄▆▄▅▅▄▄▄▄▄▄▃▃▄▄ █
  15.7 ns      Histogram: log(frequency) by time      18.7 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### N = 32

In [33]:
let N = 32
    A  = Tensor(Dense(Dense(Element(0.0))), rand(N, N))
    sA = specialize(A)
    x  = Tensor(Dense(Element(0.0)), rand(N))
    sx = specialize(x)
    y  = Tensor(Dense(Element(0.0)))
    println("── normal ──")
    display(@benchmark matvec($y, $A, $x))
    println("\n── specialized ──")
    display(@benchmark matvec_spec($y, $sA, $sx))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 925 evaluations per sample.
 Range (min … max):  111.893 ns … 143.709 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     112.411 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   112.903 ns ±   1.244 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▃▆▇██▇▆▅▄▃▂▁▁▂▃▃▃▃▂▂▂▁▁ ▁   ▁▁▂▃▂▂▁                          ▂
  ▇██████████████████████████████████████▇▇▆▇▇▇▇▇▇▇▆▆▆▇████▇▇▄▄ █
  112 ns        Histogram: log(frequency) by time        117 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 930 evaluations per sample.
 Range (min … max):  108.982 ns … 283.390 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     109.558 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   109.869 ns ±   1.985 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▃▇█▅▁                                                      
  ▁▂▃▇█████▅▃▂▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  109 ns           Histogram: frequency by time          114 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### N = 256

In [34]:
let N = 256
    A  = Tensor(Dense(Dense(Element(0.0))), rand(N, N))
    sA = specialize(A)
    x  = Tensor(Dense(Element(0.0)), rand(N))
    sx = specialize(x)
    y  = Tensor(Dense(Element(0.0)))
    println("── normal ──")
    display(@benchmark matvec($y, $A, $x))
    println("\n── specialized ──")
    display(@benchmark matvec_spec($y, $sA, $sx))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 7 evaluations per sample.
 Range (min … max):  4.765 μs …   8.726 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.791 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.820 μs ± 116.988 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄██▆▅▃▂▁▄▄▂▁   ▁▂▁▁▁▂▁                                      ▂
  █████████████▇██████████▇█▆▆▇▆▅▅▆▅▅▅▄▄▄▄▅▅▅▃▄▃▃▁▁▅▁▃▄▃▁▁▃▃▃ █
  4.76 μs      Histogram: log(frequency) by time      5.36 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 7 evaluations per sample.
 Range (min … max):  4.602 μs …   7.849 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.624 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.658 μs ± 133.945 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆█▇▄  ▁▂    ▁▁▁▂▁                                           ▁
  █████▇███▆▇▇█████▇▇▆▆▅▅▄▆▅▅▄▅▅▆▃▅▃▄▅▄▅▅▅▅▄▄▅▅▅▅▅▅▄▅▄▄▃▄▄▆▆▄ █
  4.6 μs       Histogram: log(frequency) by time      5.34 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

---
## 4. SpMV: Dense(SparseList) × Dense
### N = 100, nnz/col = 10

In [35]:
let N = 100, nnz_per_col = 10
    density = nnz_per_col / N
    A  = Tensor(Dense(SparseList(Element(0.0))), fsprand(N, N, density))
    sA = specialize(A)
    x  = Tensor(Dense(Element(0.0)), rand(N))
    y  = Tensor(Dense(Element(0.0)))
    println("── normal ──")
    display(@benchmark spmv($y, $A, $x))
    println("\n── specialized ──")
    display(@benchmark spmv_spec($y, $sA, $x))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 197 evaluations per sample.
 Range (min … max):  453.563 ns … 540.401 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     458.234 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   460.406 ns ±   6.757 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▂▄▆▇▇███▇▇▆▅▃▃▃▃▃▃▃▂▂▁▁                      ▁▁▁▁ ▁  ▁      ▃
  ▅▇████████████████████████▇▇█▇█▇▇▇▅▇█▆█▇██▇████████████████▇▆ █
  454 ns        Histogram: log(frequency) by time        485 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 198 evaluations per sample.
 Range (min … max):  460.045 ns …  1.308 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     475.510 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   476.803 ns ± 10.991 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

                 ▂▂▄▄▇▆██▇▇▆▅▄▃▂▁                               
  ▁▁▁▁▁▂▂▂▃▄▅▅▆▇██████████████████▆▆▅▅▄▃▃▃▃▃▃▃▃▃▃▃▃▂▂▃▂▂▂▂▂▂▁▂ ▄
  460 ns          Histogram: frequency by time          499 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### N = 10,000, nnz/col = 10

In [36]:
let N = 10_000, nnz_per_col = 10
    density = nnz_per_col / N
    A  = Tensor(Dense(SparseList(Element(0.0))), fsprand(N, N, density))
    sA = specialize(A)
    x  = Tensor(Dense(Element(0.0)), rand(N))
    y  = Tensor(Dense(Element(0.0)))
    println("── normal ──")
    display(@benchmark spmv($y, $A, $x))
    println("\n── specialized ──")
    display(@benchmark spmv_spec($y, $sA, $x))
end

── normal ──


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  76.046 μs … 155.030 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     77.494 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   77.956 μs ±   1.777 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▂▅▇█▇█▆▅▄▄▂                                               
  ▁▂▄█████████████▇▆▆▆▆▅▅▄▃▃▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▂▂▁▂▁▁▁▁▁▁ ▃
  76 μs           Histogram: frequency by time         83.3 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.


── specialized ──


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  75.238 μs … 141.129 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     77.079 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   77.527 μs ±   1.746 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▂▆▆▆███▆▄▄▂ ▁▁▂▃▃                                         
  ▁▂▃▆██████████████████▇▆▅▄▃▃▄▃▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁ ▄
  75.2 μs         Histogram: frequency by time           83 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

---
## 5. Compile Overhead: `execute_code` Generation

In [37]:
let
    A  = Tensor(Dense(Dense(Element(0.0))))
    sA = specialize(Tensor(Dense(Dense(Element(0.0))), rand(1, 1)))
    s  = Tensor(Element(0.0))

    println("── normal ──")
    display(@benchmark Finch.execute_code(
        :ex,
        typeof(Finch.@finch_program_instance begin
            $s .= 0
            for j in _, i in _
                $s[] += $A[i, j]
            end
            return $s
        end),
    ))

    println("\n── specialized ──")
    display(@benchmark Finch.execute_code(
        :ex,
        typeof(Finch.@finch_program_instance begin
            $s .= 0
            for j in _, i in _
                $s[] += $sA[i, j]
            end
            return $s
        end),
    ))
end

── normal ──


BenchmarkTools.Trial: 1187 samples with 1 evaluation per sample.
 Range (min … max):  3.836 ms … 27.561 ms  ┊ GC (min … max): 0.00% … 83.13%
 Time  (median):     3.945 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.209 ms ±  1.750 ms  ┊ GC (mean ± σ):  4.37% ±  8.46%

  █                                                           
  █▃▄▃▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂ ▂
  3.84 ms        Histogram: frequency by time        15.9 ms <

 Memory estimate: 2.15 MiB, allocs estimate: 65344.


── specialized ──


BenchmarkTools.Trial: 1179 samples with 1 evaluation per sample.
 Range (min … max):  3.882 ms … 30.665 ms  ┊ GC (min … max): 0.00% … 83.85%
 Time  (median):     3.980 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.237 ms ±  1.772 ms  ┊ GC (mean ± σ):  4.24% ±  8.24%

  █▂▄▁                                                        
  ████▄▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▇
  3.88 ms      Histogram: log(frequency) by time     15.8 ms <

 Memory estimate: 2.07 MiB, allocs estimate: 64037.